# Training curves — every canonical run

One plot per trained run (`runs/<topic>/<run>/metrics.jsonl`, written by `scripts/train.py`
every 5k steps): train and validation loss against training step on a log axis, with a
dashed vertical line at the step whose validation loss is lowest — the checkpoint that
`best_model.pt` holds and that every score in `build_full_table.ipynb` is computed on.

Excluded, as everywhere: `runs/archive/`, `_`-prefixed topics (baselines, smoke), and
`runs/training_curve/` (a canonical run's own checkpoints laid out as run dirs — the same
curve, not new runs). The loss is the run's own objective: MSE on discworld, token
cross-entropy on Othello, so panels are comparable only within an environment.

In [ ]:
# [1] Collect every canonical run's metrics.jsonl -> one tidy frame.
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
from pim.figures.theme import PALETTE   # Okabe-Ito, validated 2026-09-01

LOG_Y = False        # flip for log-log; the x axis is always log
SHORT = {"transformer_l": "Transformer-L", "transformer_l_tokens": "Transformer-L",
         "transformer_s": "Transformer-S", "transformer_s_tokens": "Transformer-S",
         "recurrent_l": "Recurrent-L"}
OBJECTIVE = {"discworld": "MSE (next frame)", "othello": "cross-entropy (next token)"}

rows = []
for mp in sorted((REPO / "runs").rglob("metrics.jsonl")):
    rel = mp.relative_to(REPO / "runs")
    if rel.parts[0] in ("archive", "training_curve") or rel.parts[0].startswith("_"):
        continue
    cfg = json.loads((mp.parent / "config.json").read_text())
    data = cfg.get("data", {})
    for line in mp.read_text().splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        rows.append({"topic": rel.parts[0], "run": mp.parent.name, "arch": cfg["arch"],
                     "env": data.get("env", cfg.get("env", "?")),
                     "instance": data.get("instance", "?"), "step": r["step"],
                     "train_loss": r.get("train_loss"), "val_loss": r.get("val_loss")})
M = pd.DataFrame(rows).sort_values(["env", "arch", "run", "step"]).reset_index(drop=True)
# the row order of the master tables: Othello first, transformers before recurrent
_key = lambda r: (0 if r.env == "othello" else 1, 0 if r.arch.startswith("transformer") else 1, r.run)
RUNS = sorted(M[["env", "arch", "run", "instance", "topic"]].drop_duplicates().itertuples(index=False), key=_key)
print(f"{len(RUNS)} runs · {len(M)} logged rows")
for r in RUNS:
    d = M[M["run"] == r.run]
    print(f"  {r.env:9s} {SHORT.get(r.arch, r.arch):14s} {r.run:22s} {r.instance:13s}"
          f" steps {d.step.min():>6}..{d.step.max():<7} best val {d.val_loss.min():.5f} @ {int(d.loc[d.val_loss.idxmin(), 'step'])}")

In [ ]:
# [2] One figure per run: train + val loss vs step (log x), dashed line at the best val step.
hexc = lambda i: "#%02x%02x%02x" % tuple(int(round(v * 255)) for v in PALETTE[i])
C_TRAIN, C_VAL, INK2, GRID = hexc(0), hexc(1), "#52514e", "#e1e0d9"
plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 300})
kfmt = lambda s: f"{s / 1000:g}k" if s < 1_000_000 else f"{s / 1e6:g}M"

for r in RUNS:
    d = M[M["run"] == r.run].sort_values("step")
    ib = d["val_loss"].idxmin()
    best_step, best_val = int(d.loc[ib, "step"]), float(d.loc[ib, "val_loss"])
    fig, ax = plt.subplots(figsize=(7.2, 3.9))
    ax.plot(d["step"], d["train_loss"], color=C_TRAIN, lw=1.6, label="train loss")
    ax.plot(d["step"], d["val_loss"], color=C_VAL, lw=2.0, label="val loss")
    ax.axvline(best_step, color=INK2, lw=1.2, ls="--",
               label=f"best val · step {kfmt(best_step)} · {best_val:.4g}")
    ax.set_xscale("log")
    if LOG_Y:
        ax.set_yscale("log")
    ax.set_xlabel("training step (log)", fontsize=9, color=INK2)
    ax.set_ylabel(f"loss — {OBJECTIVE.get(r.env, r.env)}", fontsize=9, color=INK2)
    ax.set_title(f"{r.run} — {SHORT.get(r.arch, r.arch)} on {r.env} · {r.instance}",
                 fontsize=10.5, loc="left", pad=8)
    ax.grid(True, which="major", color=GRID, lw=0.8)
    ax.grid(True, which="minor", axis="x", color=GRID, lw=0.4)
    ax.set_axisbelow(True)
    for sp in ax.spines.values():
        sp.set_edgecolor("#c3c2b7")
    ax.tick_params(labelsize=8, colors=INK2)
    ax.legend(fontsize=8, frameon=False, loc="best")   # clear of the divergence spikes
    plt.show()

In [ ]:
# [3] Table view: the numbers behind the dashed lines.
T = (M.groupby("run").apply(lambda d: pd.Series({
        "env": d.env.iloc[0], "arch": d.arch.iloc[0], "instance": d.instance.iloc[0],
        "first step": int(d.step.min()), "last step": int(d.step.max()),
        "best val step": int(d.loc[d.val_loss.idxmin(), "step"]),
        "best val": d.val_loss.min(), "final val": d.val_loss.iloc[-1],
        "final train": d.train_loss.iloc[-1]}), include_groups=False)
     .loc[[r.run for r in RUNS]])
display(T)